
# Amazon Reviews 2023 — Electronics Metadata EDA v1

This notebook performs the **first data-inspection stage** for the Industry Project.

## Goals

1. Download the full `meta_Electronics.jsonl.gz` file.
2. Decompress it to raw JSONL.
3. Remove the currently irrelevant top-level fields:
   - `images`
   - `videos`
   - `bought_together`
4. Create a cleaned JSONL copy.
5. Inspect what the real metadata actually looks like before defining any laptop extraction rules.
6. Examine:
   - row count
   - top-level fields
   - example records
   - field types
   - missingness
   - category values
   - nested `details`, `features`, and `description`
   - duplicate `parent_asin` behaviour in a sample

> **Important:** the source file is `jsonl.gz`. After decompression it becomes **JSONL**, not Parquet.  
> We will inspect the raw structure first and decide later how best to construct the canonical laptop catalogue and Parquet artifact.

This notebook deliberately does **not** define the final laptop filter yet. The objective is to let the real data determine the extraction rules.


In [1]:
!pip -q install -U polars pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.0 MB/s eta 0:00:00


In [2]:

import os
import json
import gzip
import shutil
import subprocess
from pathlib import Path
from collections import Counter

import pandas as pd
import polars as pl

print("pandas:", pd.__version__)
print("polars:", pl.__version__)


pandas: 2.2.3
polars: 1.44.2


## 1. Configuration

In [3]:

DATA_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Electronics.jsonl.gz"

WORKDIR = Path("/content/amazon_electronics_eda")
WORKDIR.mkdir(parents=True, exist_ok=True)

GZ_PATH = WORKDIR / "meta_Electronics.jsonl.gz"
RAW_JSONL_PATH = WORKDIR / "meta_Electronics.jsonl"
CLEAN_JSONL_PATH = WORKDIR / "meta_Electronics_cleaned.jsonl"

DROP_FIELDS = {"images", "videos", "bought_together"}
EDA_SAMPLE_ROWS = 50_000
DELETE_RAW_JSONL_AFTER_CLEANING = False

print("Working directory:", WORKDIR)
print("Drop fields:", DROP_FIELDS)


Working directory: /content/amazon_electronics_eda
Drop fields: {'images', 'bought_together', 'videos'}



## 2. Download the original compressed metadata

The file is large, so `wget -c` is used to support resume if the Colab session is interrupted.


In [4]:

if GZ_PATH.exists() and GZ_PATH.stat().st_size > 0:
    print("Compressed file already exists:", GZ_PATH)
else:
    subprocess.run(
        ["wget", "-c", DATA_URL, "-O", str(GZ_PATH)],
        check=True
    )

print("Compressed file:", GZ_PATH)
print("Size (GB):", round(GZ_PATH.stat().st_size / (1024**3), 3))


Compressed file: /content/amazon_electronics_eda/meta_Electronics.jsonl.gz
Size (GB): 1.223


## 3. Decompress to raw JSONL

In [5]:

if RAW_JSONL_PATH.exists() and RAW_JSONL_PATH.stat().st_size > 0:
    print("Raw JSONL already exists:", RAW_JSONL_PATH)
else:
    print("Decompressing...")
    with gzip.open(GZ_PATH, "rb") as src, open(RAW_JSONL_PATH, "wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)

print("Raw JSONL:", RAW_JSONL_PATH)
print("Size (GB):", round(RAW_JSONL_PATH.stat().st_size / (1024**3), 3))


Decompressing...
Raw JSONL: /content/amazon_electronics_eda/meta_Electronics.jsonl
Size (GB): 4.886



## 4. Count records

Amazon metadata is stored as one JSON object per line, so the number of lines is the number of product metadata records.


In [6]:

def count_lines_fast(path: Path) -> int:
    result = subprocess.run(
        ["wc", "-l", str(path)],
        check=True,
        capture_output=True,
        text=True
    )
    return int(result.stdout.strip().split()[0])

total_rows = count_lines_fast(RAW_JSONL_PATH)
print(f"Total metadata rows: {total_rows:,}")


Total metadata rows: 1,610,012



## 5. Inspect the original top-level structure before deleting fields

Read only a few records first. This helps confirm the raw schema before any transformation.


In [7]:

def read_first_jsonl_records(path: Path, n=5):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            records.append(json.loads(line))
    return records

raw_examples = read_first_jsonl_records(RAW_JSONL_PATH, n=5)

print("Top-level keys in first record:")
print(list(raw_examples[0].keys()))

for i, rec in enumerate(raw_examples[:3], start=1):
    print("\n" + "=" * 100)
    print(f"RAW RECORD {i}")
    print("=" * 100)
    print(json.dumps(rec, indent=2, ensure_ascii=False)[:6000])


Top-level keys in first record:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together']

RAW RECORD 1
{
  "main_category": "All Electronics",
  "title": "FS-1051 FATSHARK TELEPORTER V3 HEADSET",
  "average_rating": 3.5,
  "rating_number": 6,
  "features": [],
  "description": [
    "Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” cam


## 6. Create cleaned JSONL

This performs a streaming transformation over the full file and removes only:

- `images`
- `videos`
- `bought_together`

Everything else is preserved exactly at this stage.

The cleaning is deliberately conservative because we have not yet completed EDA.


In [8]:

def clean_jsonl_stream(
    src_path: Path,
    dst_path: Path,
    drop_fields,
    progress_every=200_000
):
    processed = 0
    malformed = 0

    with open(src_path, "r", encoding="utf-8") as src, open(dst_path, "w", encoding="utf-8") as dst:
        for line in src:
            try:
                obj = json.loads(line)
                for field in drop_fields:
                    obj.pop(field, None)

                dst.write(json.dumps(obj, ensure_ascii=False, separators=(",", ":")) + "\n")
                processed += 1

                if processed % progress_every == 0:
                    print(f"Processed {processed:,} rows")

            except json.JSONDecodeError:
                malformed += 1

    return processed, malformed


if CLEAN_JSONL_PATH.exists() and CLEAN_JSONL_PATH.stat().st_size > 0:
    print("Cleaned JSONL already exists:", CLEAN_JSONL_PATH)
else:
    processed_rows, malformed_rows = clean_jsonl_stream(
        RAW_JSONL_PATH,
        CLEAN_JSONL_PATH,
        DROP_FIELDS
    )
    print(f"Processed rows: {processed_rows:,}")
    print(f"Malformed rows skipped: {malformed_rows:,}")

print("Cleaned JSONL:", CLEAN_JSONL_PATH)
print("Size (GB):", round(CLEAN_JSONL_PATH.stat().st_size / (1024**3), 3))


Processed 200,000 rows
Processed 400,000 rows
Processed 600,000 rows
Processed 800,000 rows
Processed 1,000,000 rows
Processed 1,200,000 rows
Processed 1,400,000 rows
Processed 1,600,000 rows
Processed rows: 1,610,012
Malformed rows skipped: 0
Cleaned JSONL: /content/amazon_electronics_eda/meta_Electronics_cleaned.jsonl
Size (GB): 2.6


## 7. Verify removed fields

In [9]:

clean_examples = read_first_jsonl_records(CLEAN_JSONL_PATH, n=5)

for field in DROP_FIELDS:
    assert all(field not in rec for rec in clean_examples), f"{field} still present"

print("Removed fields verified:", sorted(DROP_FIELDS))
print("\nRemaining top-level keys:")
print(list(clean_examples[0].keys()))


Removed fields verified: ['bought_together', 'images', 'videos']

Remaining top-level keys:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'store', 'categories', 'details', 'parent_asin']



## 8. Display a compact human-readable sample

Instead of printing every nested field, show the fields most likely to matter for later laptop EDA.


In [10]:

def compact_record(rec):
    return {
        "parent_asin": rec.get("parent_asin"),
        "title": rec.get("title"),
        "main_category": rec.get("main_category"),
        "categories": rec.get("categories"),
        "store": rec.get("store"),
        "price": rec.get("price"),
        "average_rating": rec.get("average_rating"),
        "rating_number": rec.get("rating_number"),
        "features": rec.get("features"),
        "description": rec.get("description"),
        "details": rec.get("details"),
    }

compact_examples = [compact_record(r) for r in clean_examples]
sample_df = pd.DataFrame(compact_examples)

pd.set_option("display.max_colwidth", 250)
display(sample_df)


,parent_asin,title,main_category,categories,store,price,average_rating,rating_number,features,description,details
0,B00MCW7G9M,FS-1051 FATSHARK TELEPORTER V3 HEADSET,All Electronics,"[Electronics, Television & Video, Video Glasses]",Fat Shark,NaN,3.5,6,[],[Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver tec...,"{'Date First Available': 'August 2, 2014', 'Manufacturer': 'Fatshark'}"
1,B00YT6XQSE,Ce-H22B12-S1 4Kx2K Hdmi 4Port,All Electronics,"[Electronics, Television & Video, Accessories, Cables, HDMI Cables]",SIIG,NaN,5.0,1,"[UPC: 662774021904, Weight: 0.600 lbs]",[HDMI In - HDMI Out],"{'Product Dimensions': '0.83 x 4.17 x 2.05 inches', 'Item Weight': '5.3 ounces', 'Item model number': 'CE-H22B12-S1', 'Is Discontinued By Manufacturer': 'No', 'Date First Available': 'June 3, 2015', 'Manufacturer': 'SIIG'}"
2,B07SM135LS,"Digi-Tatoo Decal Skin Compatible With MacBook Pro 13 inch (Model A2338/ A2289/ A2251) - Protective and Decorative Full Body Laptop Skin Decal Sticker, Anti-Scratch Vinly Skin Sticker Wrap [Fresh Marble]",Computers,"[Electronics, Computers & Accessories, Laptop Accessories, Skins & Decals, Decals]",Digi-Tatoo,19.99,4.5,246,"[WARNING: Please IDENTIFY MODEL NUMBER on the bottom of your Macbook. Only fits for model A2338/ A2289/ A2251 (Macbook Pro 13"" w/ Touch Bar, 2022/2020 release)., Extra Care Yet Not Bulky. Our skin is capable of protecting the surface of your Macb...",[],"{'Brand': 'Digi-Tatoo', 'Color': 'Fresh Marble', 'Room Type': 'Bedroom', 'Material': 'Vinyl', 'Theme': 'Marble', 'Style': 'fresh marble', 'Age Range (Description)': 'Adult', 'Reusability': 'Single Use', 'Pattern': 'Marble', 'Shape': 'Rectangular'..."
3,B089CNGZCW,NotoCity Compatible with Vivoactive 4 band 22mm Quick Release Silicone Bands/Garmin Darth Vader/First Avenger/Polar Vantage Smartwatch Sport Breathable Strap Replacement for Gear S3 Classic Watchband,AMAZON FASHION,"[Electronics, Wearable Technology, Clips, Arm & Wristbands]",NotoCity,9.99,4.5,233,[☛NotoCity 22mm band is designed for Vivoactive 4 / Samsung Gear S3 Classic / S3 Frontier / Gear Live / Samsung Galaxy Watch 46mm / Huawei Watch GT / Tickwatch Pro / S2 / E2 / Pebble Classic / Time / Time Steel / Pebble 2 (Not Pebble steel) / Oth...,[],"{'Date First Available': 'May 29, 2020', 'Manufacturer': 'NotoCity'}"
4,B004E2Z88O,Motorola Droid X Essentials Combo Pack,Cell Phones & Accessories,"[Electronics, Computers & Accessories, Computer Accessories & Peripherals, Memory Cards, Micro SD Cards]",Verizon,14.99,3.8,64,"[New Droid X Essentials Combo Pack, Exclusive Package Incredible Value Worth $145!!!, Includes all Genuine High Quality Motorola Made Accessories]","[all Genuine High Quality Motorola Made Accessories, including Multimedia Station with HDMI technology, HDMI Cable and AC Wall Charger, Motorola Navigation / Music Vehicle Charging Mount Car Dock and Motorola 12v Vehicle Power Adapter Charger!]","{'Product Dimensions': '11.6 x 6.9 x 3.1 inches', 'Item Weight': '1.5 pounds', 'Other display features': 'Wireless', 'Manufacturer': 'Verizon', 'Date First Available': 'November 26, 2010'}"



## 9. Load a 50k-row EDA sample

We do not need to load the entire Electronics dataset into RAM just to understand its structure.

This sample is used for:
- field coverage
- type inspection
- category exploration
- nested-field examples
- duplicate checks

Later laptop extraction will operate on the complete dataset.


In [11]:

def load_jsonl_sample(path: Path, n: int):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            rows.append(json.loads(line))
    return rows

eda_records = load_jsonl_sample(CLEAN_JSONL_PATH, EDA_SAMPLE_ROWS)
print(f"Loaded EDA sample: {len(eda_records):,} rows")


Loaded EDA sample: 50,000 rows


## 10. Top-level field coverage and observed Python types

In [12]:

all_keys = sorted(set().union(*(r.keys() for r in eda_records)))

coverage_rows = []

for key in all_keys:
    values = [r.get(key) for r in eda_records]
    non_null = [v for v in values if v is not None]
    observed_types = sorted({type(v).__name__ for v in non_null})

    coverage_rows.append({
        "field": key,
        "non_null_count": len(non_null),
        "coverage_pct": round(100 * len(non_null) / len(eda_records), 2),
        "observed_types": ", ".join(observed_types)
    })

coverage_df = (
    pd.DataFrame(coverage_rows)
    .sort_values(["coverage_pct", "field"], ascending=[False, True])
    .reset_index(drop=True)
)

display(coverage_df)


,field,non_null_count,coverage_pct,observed_types
0,average_rating,50000,100.00,float
1,categories,50000,100.00,list
2,description,50000,100.00,list
3,details,50000,100.00,dict
4,features,50000,100.00,list
5,parent_asin,50000,100.00,str
6,rating_number,50000,100.00,int
7,title,50000,100.00,str
8,store,49725,99.45,str
9,main_category,49428,98.86,str


## 11. Basic numeric / scalar summaries

In [13]:

scalar_fields = ["price", "average_rating", "rating_number"]
scalar_summary = []

for field in scalar_fields:
    vals = [r.get(field) for r in eda_records if r.get(field) is not None]
    numeric = pd.to_numeric(pd.Series(vals), errors="coerce").dropna()

    scalar_summary.append({
        "field": field,
        "non_null": len(vals),
        "numeric_parseable": len(numeric),
        "min": numeric.min() if len(numeric) else None,
        "median": numeric.median() if len(numeric) else None,
        "mean": numeric.mean() if len(numeric) else None,
        "max": numeric.max() if len(numeric) else None,
    })

display(pd.DataFrame(scalar_summary))


,field,non_null,numeric_parseable,min,median,mean,max
0,price,21412,21412,0.01,20.25,87.681514,12998.0
1,average_rating,50000,50000,1.00,4.20,4.076738,5.0
2,rating_number,50000,50000,1.00,21.00,500.742040,315583.0



## 12. Explore category values

This is one of the most important cells for the next stage.

We want to know whether Amazon already provides useful category labels that can identify laptops more reliably than title-only filtering.


In [14]:

category_counter = Counter()
main_category_counter = Counter()

for rec in eda_records:
    main_cat = rec.get("main_category")
    if main_cat:
        main_category_counter[str(main_cat)] += 1

    categories = rec.get("categories")
    if isinstance(categories, list):
        for c in categories:
            if c is not None:
                category_counter[str(c)] += 1

print("Top main_category values:")
display(pd.DataFrame(main_category_counter.most_common(30), columns=["main_category", "count"]))

print("\nTop categories values:")
display(pd.DataFrame(category_counter.most_common(100), columns=["category", "count"]))


Top main_category values:


,main_category,count
0,Computers,14610
1,All Electronics,12391
2,Camera & Photo,6571
3,Cell Phones & Accessories,4899
4,Home Audio & Theater,3225
5,Industrial & Scientific,1616
6,AMAZON FASHION,801
7,Car Electronics,773
8,Tools & Home Improvement,764
9,Amazon Home,746



Top categories values:


,category,count
0,Electronics,46300
1,Computers & Accessories,20968
2,Camera & Photo,7472
3,Accessories,5874
4,"Bags, Cases & Sleeves",5631
...,...,...
95,Camera Batteries,267
96,Power Strips & Surge Protectors,265
97,Monitor Accessories,258
98,Towers,254


## 13. Search category values for laptop-related terms — exploratory only

In [15]:

laptop_terms = ["laptop", "notebook", "chromebook", "macbook", "computer"]

category_matches = [
    (category, count)
    for category, count in category_counter.items()
    if any(term in category.lower() for term in laptop_terms)
]

category_matches_df = pd.DataFrame(
    sorted(category_matches, key=lambda x: x[1], reverse=True),
    columns=["category", "count"]
)

display(category_matches_df.head(100))


,category,count
0,Computers & Accessories,20968
1,Computer Accessories & Peripherals,5455
2,Laptop Accessories,4484
3,Computer Components,2823
4,Computers & Tablets,1661
5,Laptops,923
6,Traditional Laptops,849
7,Computer Cable Adapters,403
8,Laptop Replacement Parts,268
9,Computer Cases,160



## 14. Inspect nested fields: `details`, `features`, `description`

These fields are likely to contain useful laptop attributes later, but we should first inspect how consistent they are.


In [16]:

def show_nonempty_examples(field, n=10):
    shown = 0

    for rec in eda_records:
        value = rec.get(field)

        if value not in (None, "", [], {}):
            print("=" * 100)
            print("TITLE:", rec.get("title"))
            print(f"{field.upper()}:")
            print(json.dumps(value, indent=2, ensure_ascii=False)[:4000])

            shown += 1
            if shown >= n:
                break

print("DETAILS EXAMPLES")
show_nonempty_examples("details", n=8)

print("\nFEATURES EXAMPLES")
show_nonempty_examples("features", n=8)

print("\nDESCRIPTION EXAMPLES")
show_nonempty_examples("description", n=8)


DETAILS EXAMPLES
TITLE: FS-1051 FATSHARK TELEPORTER V3 HEADSET
DETAILS:
{
  "Date First Available": "August 2, 2014",
  "Manufacturer": "Fatshark"
}
TITLE: Ce-H22B12-S1 4Kx2K Hdmi 4Port
DETAILS:
{
  "Product Dimensions": "0.83 x 4.17 x 2.05 inches",
  "Item Weight": "5.3 ounces",
  "Item model number": "CE-H22B12-S1",
  "Is Discontinued By Manufacturer": "No",
  "Date First Available": "June 3, 2015",
  "Manufacturer": "SIIG"
}
TITLE: Digi-Tatoo Decal Skin Compatible With MacBook Pro 13 inch (Model A2338/ A2289/ A2251) - Protective and Decorative Full Body Laptop Skin Decal Sticker, Anti-Scratch Vinly Skin Sticker Wrap [Fresh Marble]
DETAILS:
{
  "Brand": "Digi-Tatoo",
  "Color": "Fresh Marble",
  "Room Type": "Bedroom",
  "Material": "Vinyl",
  "Theme": "Marble",
  "Style": "fresh marble",
  "Age Range (Description)": "Adult",
  "Reusability": "Single Use",
  "Pattern": "Marble",
  "Shape": "Rectangular",
  "Unit Count": "1.0 Count",
  "Finish Type": "Powder Coated",
  "Item Weight": 


## 15. Most common keys inside `details`

`details` is usually a nested dictionary. Counting its keys can tell us whether fields such as brand, model, memory, screen size, operating system, and processor appear consistently enough to support a canonical laptop schema.


In [17]:

details_key_counter = Counter()
details_records = 0

for rec in eda_records:
    details = rec.get("details")

    if isinstance(details, dict):
        details_records += 1
        for key in details.keys():
            details_key_counter[str(key)] += 1

details_keys_df = pd.DataFrame(
    [
        {
            "details_key": key,
            "count": count,
            "coverage_within_details_pct": round(100 * count / details_records, 2)
        }
        for key, count in details_key_counter.most_common()
    ]
)

print(f"Records with details dict: {details_records:,}")
display(details_keys_df.head(100))


Records with details dict: 50,000


,details_key,count,coverage_within_details_pct
0,Date First Available,48292,96.58
1,Manufacturer,44131,88.26
2,Item Weight,42835,85.67
3,Brand,39076,78.15
4,Item model number,31357,62.71
...,...,...,...
95,Theme,663,1.33
96,Number Of Items,655,1.31
97,Hard Drive Rotational Speed,652,1.30
98,Hard Disk Description,637,1.27


## 16. `parent_asin` uniqueness in the EDA sample

In [18]:

parent_asins = [
    r.get("parent_asin")
    for r in eda_records
    if r.get("parent_asin") is not None
]

parent_counts = Counter(parent_asins)
duplicate_parents = {k: v for k, v in parent_counts.items() if v > 1}

print(f"Sample rows: {len(eda_records):,}")
print(f"Non-null parent_asin: {len(parent_asins):,}")
print(f"Unique parent_asin: {len(parent_counts):,}")
print(f"parent_asin values occurring >1 time: {len(duplicate_parents):,}")

if duplicate_parents:
    display(
        pd.DataFrame(
            sorted(duplicate_parents.items(), key=lambda x: x[1], reverse=True)[:30],
            columns=["parent_asin", "count"]
        )
    )


Sample rows: 50,000
Non-null parent_asin: 50,000
Unique parent_asin: 50,000
parent_asin values occurring >1 time: 0



## 17. Quick title exploration — NOT the final laptop filter

This is only to compare title keywords with the category information discovered above.

Do not treat this as the final extraction rule.


In [19]:

title_keywords = ["laptop", "notebook", "chromebook", "macbook", "ultrabook"]

title_matches = []

for rec in eda_records:
    title = str(rec.get("title") or "")
    title_lower = title.lower()

    if any(keyword in title_lower for keyword in title_keywords):
        title_matches.append({
            "parent_asin": rec.get("parent_asin"),
            "title": title,
            "categories": rec.get("categories"),
            "price": rec.get("price"),
            "store": rec.get("store"),
        })

title_matches_df = pd.DataFrame(title_matches)

print(f"Title-keyword matches in {len(eda_records):,}-row sample: {len(title_matches_df):,}")
display(title_matches_df.head(100))


Title-keyword matches in 50,000-row sample: 7,134


,parent_asin,title,categories,price,store
0,B07SM135LS,"Digi-Tatoo Decal Skin Compatible With MacBook Pro 13 inch (Model A2338/ A2289/ A2251) - Protective and Decorative Full Body Laptop Skin Decal Sticker, Anti-Scratch Vinly Skin Sticker Wrap [Fresh Marble]","[Electronics, Computers & Accessories, Laptop Accessories, Skins & Decals, Decals]",19.99,Digi-Tatoo
1,B01MCZP7RF,"MOSISO Plastic Hard Shell Case & Keyboard Cover Compatible MacBook Air 11 Inch (Models: A1370 & A1465), Transparent Black","[Electronics, Headphones, Earbuds & Accessories, Adapters]",NaN,MOSISO
2,B07Y6MRH7G,"KEiiD PC Computer Speaker Compact Bluetooth Stereo System with Aluminum Housing for Laptop Desktop PC Gaming Computers, Unique Touch Controller, Built-in Bluetooth 5.0 Receiver for Wireless Streaming","[Electronics, Computers & Accessories, Computer Accessories & Peripherals, Audio & Video Accessories, Computer Speakers]",NaN,KEiiD
3,B0BPLX8B2K,"Outer Space Planets Stickers(50Pcs),Planetary Systems Waterproof Vinyl Water Bottle Stickers for Teens Girls Adults Kids,Laptop Car Cup Computer Guitar Skateboard Luggage Phone Aesthetic Stickers Pack","[Electronics, Computers & Accessories, Laptop Accessories, Skins & Decals, Decals]",3.99,Affoto
4,B0BYBG1PPD,"Gateway 15.6"" FHD Ultra Slim Budget Notebook, Intel Pentium Processor(4-Core, up to 3.1 GHz), 4GB RAM, 128GB Storage, Precision Touchpad, HDMI, Type-C Windows 10S, Microsoft 365 1 Year, Charcoal Black","[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]",189.99,Gateway
...,...,...,...,...,...
95,B09432ZFQ6,"Acer Aspire 3 Laptop, 17.3'' HD+ 1600 x 900, 3.60GHz Intel Core i5-1035G, 20GB DDR4 RAM, 128GB SSD + 1TB HDD, DVD-Writer, Windows 10, A317-52","[Electronics, Computers & Accessories, Computers & Tablets, Laptops, Traditional Laptops]",539.99,Acer
96,B07KRRW1PC,"[Upgraded] Smart&Cool Gen8 Max 100W USB-C to USB-C Charging & Data Sync Magnetic Cable with Power Delivery, Compatible with Laptops and Mobile Devices with USB-C Interface (Silver-2Pack, 6 Feet)","[Electronics, Computers & Accessories, Computer Accessories & Peripherals, Cables & Accessories, Cables & Interconnects, USB Cables]",NaN,Smart&Cool
97,B09DCFC9HM,KECC Compatible with MacBook Pro 13 inch Case 2016-2020 Release A2159 A1989 A1706 A1708 Touch Bar Plastic Hard Shell + Keyboard Cover + Sleeve + Screen Protector (Baby Pink Sparkling),"[Electronics, Computers & Accessories, Laptop Accessories, Bags, Cases & Sleeves, Hard Shell Cases]",29.99,KECC
98,B01BW8SM0S,"Josi Minea USB 3.1 Type C Male (USB-C) to USB 3.0 Type A Female OTG Host Adapter for Samsung Galaxy S8 & S8 Plus, Google Pixel, GoPro HERO5, LG G6 & G5, Nexus 5X / 6P, New Apple MacBook & OnePlus 3/2",[],NaN,Josi Minea


## 18. Optional disk cleanup

In [20]:

if DELETE_RAW_JSONL_AFTER_CLEANING:
    if CLEAN_JSONL_PATH.exists() and CLEAN_JSONL_PATH.stat().st_size > 0:
        RAW_JSONL_PATH.unlink(missing_ok=True)
        print("Deleted raw uncompressed JSONL:", RAW_JSONL_PATH)
    else:
        print("Cleaned file not found; raw JSONL was not deleted.")
else:
    print("DELETE_RAW_JSONL_AFTER_CLEANING=False; raw JSONL retained.")


DELETE_RAW_JSONL_AFTER_CLEANING=False; raw JSONL retained.



# What to send back after running this notebook

The most useful outputs for designing the laptop extraction pipeline are:

1. **Total row count**
2. **Remaining top-level keys**
3. **Field coverage table**
4. **Top category values**
5. **Laptop-related category search results**
6. **A few examples of `details` from likely laptop records**
7. **Most common `details` keys**
8. **Title-keyword match examples**

After seeing those results, the next notebook can define:

**Electronics → laptop candidate extraction → accessory exclusion → canonical laptop schema → normalization → final clean catalogue**

without guessing how the raw Amazon metadata is structured.


## Save Original Data

In [21]:
# Mount my drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [22]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/UTS Courses/Industrial Project"
)

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_DIR)

/content/drive/MyDrive/UTS Courses/Industrial Project


In [23]:
import shutil

src = Path(
    "/content/amazon_electronics_eda/meta_Electronics_cleaned.jsonl"
)

dst = PROJECT_DIR / "meta_Electronics_cleaned.jsonl"

shutil.copy2(src, dst)

print("Saved to:", dst)
print("Size GB:", round(dst.stat().st_size / (1024**3), 3))

Saved to: /content/drive/MyDrive/UTS Courses/Industrial Project/meta_Electronics_cleaned.jsonl
Size GB: 2.6
